In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
pd.set_option('display.max_columns', 500)

## 1. Inspecting the dataset

In [3]:
path = '../raw/dirty'

In [4]:
dirty = pd.read_csv(path+'/diabetes-dirty.csv', sep=';')

In [5]:
dirty.head()

,Unnamed: 0,Index,Pregnancies,Glucose,Blood Pressure,Skin Thickness,Insulin,Body Mass Index,Diabetes Pedigree Function,Age,Outcome,BMI Category,Clinic Region,Care Path,Patient Segment
0,0,0,6,148.0 mg/dL,72.0,35.0,nan uU/mL,33.6 bmi,0.627,50.0 years,positive,obese,north,high-risk,adult
1,1,1,1,85.0,66.0,29.0,NaN,26.6,0.351,31.0,negative,OVERWEIGHT,SOUTH,ROUTINE FOLLOW-UP,SENIOR
2,2,2,8,183.0,64.0,NaN,NaN,23.3,0.672,32.0,1,normal-weight,N,urgent monitoring,mid-age
3,3,3,1,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0,under wt,south-side,routine,?
4,4,4,0,137.0,40.0,35.0,168.0,43.1,2.288,33.0,1,unknown,?,NaN,Adult


In [6]:
dirty.columns

Index(['Unnamed: 0', ' Index ', 'Pregnancies ', ' Glucose', 'Blood Pressure',
       'Skin Thickness ', 'Insulin', 'Body Mass Index',
       'Diabetes Pedigree Function', 'Age ', 'Outcome ', 'BMI Category',
       'Clinic Region', 'Care Path ', 'Patient Segment'],
      dtype='str')

In [7]:
dirty = dirty.drop(columns=['Unnamed: 0', ' Index '])

In [8]:
dirty.shape

(771, 13)

In [9]:
dirty.info()

<class 'pandas.DataFrame'>
RangeIndex: 771 entries, 0 to 770
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Pregnancies                 771 non-null    int64  
 1    Glucose                    714 non-null    str    
 2   Blood Pressure              666 non-null    float64
 3   Skin Thickness              543 non-null    float64
 4   Insulin                     425 non-null    str    
 5   Body Mass Index             718 non-null    str    
 6   Diabetes Pedigree Function  771 non-null    float64
 7   Age                         709 non-null    str    
 8   Outcome                     744 non-null    str    
 9   BMI Category                686 non-null    str    
 10  Clinic Region               744 non-null    str    
 11  Care Path                   721 non-null    str    
 12  Patient Segment             744 non-null    str    
dtypes: float64(3), int64(1), str(9)
memory usage: 

In [10]:
dirty.describe()

,Pregnancies,Blood Pressure,Skin Thickness,Diabetes Pedigree Function
count,771.000000,666.000000,543.000000,771.000000
mean,3.836576,72.231231,29.116022,0.471516
std,3.373966,12.294530,10.478557,0.330866
min,-2.000000,24.000000,7.000000,0.078000
25%,1.000000,64.000000,22.000000,0.243500
50%,3.000000,72.000000,29.000000,0.374000
75%,6.000000,80.000000,36.000000,0.625000
max,17.000000,122.000000,99.000000,2.420000


## 2. Preprocessing

In [11]:
dirty.columns = [col.strip() for col in dirty.columns]
dirty.columns

Index(['Pregnancies', 'Glucose', 'Blood Pressure', 'Skin Thickness', 'Insulin',
       'Body Mass Index', 'Diabetes Pedigree Function', 'Age', 'Outcome',
       'BMI Category', 'Clinic Region', 'Care Path', 'Patient Segment'],
      dtype='str')

In [12]:
cat_list = list(dirty.select_dtypes(include='object').columns)
cat_list

C:\Users\aikyu\AppData\Local\Temp\ipykernel_5060\3528442749.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_list = list(dirty.select_dtypes(include='object').columns)


['Glucose',
 'Insulin',
 'Body Mass Index',
 'Age',
 'Outcome',
 'BMI Category',
 'Clinic Region',
 'Care Path',
 'Patient Segment']

In [13]:
num_list = list(dirty.select_dtypes(include='number').columns)
num_list

['Pregnancies',
 'Blood Pressure',
 'Skin Thickness',
 'Diabetes Pedigree Function']

In [14]:
(dirty.isna().sum()/len(dirty)*100).sort_values(ascending=False)

Insulin                       44.876783
Skin Thickness                29.571984
Blood Pressure                13.618677
BMI Category                  11.024643
Age                            8.041505
Glucose                        7.392996
Body Mass Index                6.874189
Care Path                      6.485084
Clinic Region                  3.501946
Patient Segment                3.501946
Outcome                        3.501946
Pregnancies                    0.000000
Diabetes Pedigree Function     0.000000
dtype: float64

In [15]:
(dirty[num_list].isna().sum()/len(dirty)*100).sort_values(ascending=False)

Skin Thickness                29.571984
Blood Pressure                13.618677
Pregnancies                    0.000000
Diabetes Pedigree Function     0.000000
dtype: float64

## 3. Standardizing categorical values

In [16]:
for c in cat_list:
    print(c, dirty[c].unique(), end='\n\n')

Glucose <StringArray>
['148.0 mg/dL',        '85.0',       '183.0',        '89.0',       '137.0',
          ' ?',        '78.0',       '115.0',       '197.0',       '125.0',
 ...
 '136.0 mg/dL',       '167.0', '195.0 mg/dL',       '169.0',       '366.0',
       '375.0',       '402.0',        '65.0',       '459.0',       '190.0']
Length: 175, dtype: str

Insulin <StringArray>
['nan uU/mL',         nan,      '94.0',     '168.0',        ' ?',      '88.0',
     '543.0',     '846.0',     '175.0',     '230.0',
 ...
     '387.0',      '22.0',    '1455.0',     '392.0',     '178.0',     '127.0',
     '370.0',     '510.0',      '16.0',     '112.0']
Length: 195, dtype: str

Body Mass Index <StringArray>
['33.6 bmi',     '26.6',     '23.3',   ' 28.1 ',     '43.1',       ' ?',
     '31.0',     '35.3',   ' 30.5 ',        nan,
 ...
   ' 28.5 ',    '101.5',     '49.3',   ' 46.3 ',   ' 31.2 ',     '97.5',
   ' 43.3 ',   ' 36.3 ',     '37.5',   ' 28.4 ']
Length: 340, dtype: str

Age <StringArray>
[ '50.

In [17]:
replacements = {
    '1 ': '1',
    'positive': '1',
    'negative': '0',
    ' negative ': '0',
    '?': np.nan,
    'normal-weight': 'normal',
    'under wt': 'underweight',
    'unknown': np.nan,
    ' nan ': np.nan,
    'NAN': np.nan,
    ' NAN ': np.nan,
    'north ': 'North',
    'SOUTH': 'South',
    'N': 'North',
    'south-side': 'South',
    ' ?': np.nan,
    'High Risk': 'high-risk'
}

In [18]:
for c in cat_list:
    dirty[c] = dirty[c].replace(replacements)
    print(c, dirty[c].unique(), end='\n\n')

Glucose <StringArray>
['148.0 mg/dL',        '85.0',       '183.0',        '89.0',       '137.0',
           nan,        '78.0',       '115.0',       '197.0',       '125.0',
 ...
 '136.0 mg/dL',       '167.0', '195.0 mg/dL',       '169.0',       '366.0',
       '375.0',       '402.0',        '65.0',       '459.0',       '190.0']
Length: 174, dtype: str

Insulin <StringArray>
['nan uU/mL',         nan,      '94.0',     '168.0',      '88.0',     '543.0',
     '846.0',     '175.0',     '230.0',      '83.0',
 ...
     '387.0',      '22.0',    '1455.0',     '392.0',     '178.0',     '127.0',
     '370.0',     '510.0',      '16.0',     '112.0']
Length: 193, dtype: str

Body Mass Index <StringArray>
['33.6 bmi',     '26.6',     '23.3',   ' 28.1 ',     '43.1',        nan,
     '31.0',     '35.3',   ' 30.5 ',     '37.6',
 ...
   ' 28.5 ',    '101.5',     '49.3',   ' 46.3 ',   ' 31.2 ',     '97.5',
   ' 43.3 ',   ' 36.3 ',     '37.5',   ' 28.4 ']
Length: 338, dtype: str

Age <StringArray>
[ '50.

In [19]:
to_lower = ['BMI Category', 'Care Path', 'Patient Segment']

In [20]:
for c in to_lower:
    dirty[c] = dirty[c].str.strip().str.lower()
    print(c, dirty[c].unique(), end='\n\n')

BMI Category <StringArray>
['obese', 'overweight', 'normal', 'underweight', nan]
Length: 5, dtype: str

Care Path <StringArray>
['high-risk', 'routine follow-up', 'urgent monitoring', 'routine', nan]
Length: 5, dtype: str

Patient Segment <StringArray>
['adult', 'senior', 'mid-age', nan]
Length: 4, dtype: str



## 4. Fixing column types

In [21]:
shouldbe_num = ['Glucose', 'Insulin', 'Body Mass Index', 'Age']

In [22]:
for c in shouldbe_num:
    dirty[c] = pd.to_numeric(dirty[c].astype(str).str.split(' ').str[0], errors='coerce')
    dirty[c] = dirty[c].fillna(dirty[c].median())
    print(c, dirty[c].unique(), end='\n\n')

Glucose [148.  85. 183.  89. 137. 119.  78. 115. 197. 125. 999. 168. 139. 189.
 166. 100. 118. 103. 126.  99. 196. 143. 147. 145. 117. 109. 158.  88.
  92. 122. 138. 102.  90. 111. 180. 133. 106. 171. 159.  71. 105. 528.
 150.  73. 187. 146.  84. 399.  44. 141. 114. 129.  62.  95. 131. 112.
 113.  74.  83. 101. 110. 136. 107.  80. 123.  81. 134. 142. 144. 279.
 163. 151.  96. 155.  76. 160. 124.  97. 162. 132. 120. 173. 170.  93.
 384. 108. 154.  57. 156. 153. 188. 152. 104.  87.  79.  75. 130. 194.
 181. 128. 135. 324. 184. 179. 140. 177. 513. 543.  91. 417. 165.  86.
 191. 161. 501.  77. 182. 456. 157. 178. 116.  61. 270.  98. 127.  82.
 193.  72. 172.  94. 175. 195.  68. 252. 186. 164. 567. 198. 121. 363.
 176.  67. 174. 309. 167. 169. 366. 375. 402.  65. 459. 190.]

Insulin [ 126.   94.  168.   88.  543.  846.  175.  230.   83.   96.  146.  115.
  140.  110.  245.   54.  192.  207.   70.  240.   82.   36.   23. 1500.
  342.  304.  142.  128.  100.   90.  270.   71.  125.  176.   48

In [23]:
dirty.info()

<class 'pandas.DataFrame'>
RangeIndex: 771 entries, 0 to 770
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Pregnancies                 771 non-null    int64  
 1   Glucose                     771 non-null    float64
 2   Blood Pressure              666 non-null    float64
 3   Skin Thickness              543 non-null    float64
 4   Insulin                     771 non-null    float64
 5   Body Mass Index             771 non-null    float64
 6   Diabetes Pedigree Function  771 non-null    float64
 7   Age                         771 non-null    float64
 8   Outcome                     728 non-null    str    
 9   BMI Category                650 non-null    str    
 10  Clinic Region               724 non-null    str    
 11  Care Path                   721 non-null    str    
 12  Patient Segment             703 non-null    str    
dtypes: float64(7), int64(1), str(5)
memory usage: 

## 5. Missing value imputation

In [24]:
cat_list = list(set(cat_list) - set(shouldbe_num))
num_list = num_list + shouldbe_num
cat_list, num_list

(['Care Path', 'Outcome', 'BMI Category', 'Clinic Region', 'Patient Segment'],
 ['Pregnancies',
  'Blood Pressure',
  'Skin Thickness',
  'Diabetes Pedigree Function',
  'Glucose',
  'Insulin',
  'Body Mass Index',
  'Age'])

In [25]:
(dirty[num_list].isna().sum()/len(dirty)*100).sort_values(ascending=False)

Skin Thickness                29.571984
Blood Pressure                13.618677
Pregnancies                    0.000000
Diabetes Pedigree Function     0.000000
Glucose                        0.000000
Insulin                        0.000000
Body Mass Index                0.000000
Age                            0.000000
dtype: float64

In [26]:
dirty['Skin Thickness'] = dirty['Skin Thickness'].fillna(dirty['Skin Thickness'].median())
dirty['Blood Pressure'] = dirty['Blood Pressure'].fillna(dirty['Blood Pressure'].median())

In [27]:
(dirty[num_list].isna().sum()/len(dirty)*100).sort_values(ascending=False)

Pregnancies                   0.0
Blood Pressure                0.0
Skin Thickness                0.0
Diabetes Pedigree Function    0.0
Glucose                       0.0
Insulin                       0.0
Body Mass Index               0.0
Age                           0.0
dtype: float64

In [28]:
(dirty[cat_list].isna().sum()/len(dirty)*100).sort_values(ascending=False)

BMI Category       15.693904
Patient Segment     8.819715
Care Path           6.485084
Clinic Region       6.095979
Outcome             5.577173
dtype: float64

In [29]:
for c in cat_list:
    dirty[c] = dirty[c].fillna('unknown')

In [30]:
(dirty[cat_list].isna().sum()/len(dirty)*100).sort_values(ascending=False)

Care Path          0.0
Outcome            0.0
BMI Category       0.0
Clinic Region      0.0
Patient Segment    0.0
dtype: float64

In [31]:
dirty.info()

<class 'pandas.DataFrame'>
RangeIndex: 771 entries, 0 to 770
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Pregnancies                 771 non-null    int64  
 1   Glucose                     771 non-null    float64
 2   Blood Pressure              771 non-null    float64
 3   Skin Thickness              771 non-null    float64
 4   Insulin                     771 non-null    float64
 5   Body Mass Index             771 non-null    float64
 6   Diabetes Pedigree Function  771 non-null    float64
 7   Age                         771 non-null    float64
 8   Outcome                     771 non-null    str    
 9   BMI Category                771 non-null    str    
 10  Clinic Region               771 non-null    str    
 11  Care Path                   771 non-null    str    
 12  Patient Segment             771 non-null    str    
dtypes: float64(7), int64(1), str(5)
memory usage: 

## 6. Duplicates and outliers

In [32]:
dirty.duplicated().sum()

np.int64(3)

In [33]:
dirty[dirty.duplicated(keep=False)] #???

,Pregnancies,Glucose,Blood Pressure,Skin Thickness,Insulin,Body Mass Index,Diabetes Pedigree Function,Age,Outcome,BMI Category,Clinic Region,Care Path,Patient Segment
3,1,89.0,66.0,23.0,94.0,32.4,0.167,21.0,0,underweight,South,routine,unknown
15,7,100.0,72.0,29.0,126.0,30.0,0.484,32.0,1,overweight,South,high-risk,adult
27,1,119.0,72.0,15.0,140.0,23.2,0.487,22.0,0,obese,South,routine follow-up,adult
768,1,89.0,66.0,23.0,94.0,32.4,0.167,21.0,0,underweight,South,routine,unknown
769,7,100.0,72.0,29.0,126.0,30.0,0.484,32.0,1,overweight,South,high-risk,adult
770,1,119.0,72.0,15.0,140.0,23.2,0.487,22.0,0,obese,South,routine follow-up,adult


In [34]:
def verify_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    outliers = (df[col] < (q1 - 1.5 * iqr)) | (df[col] > (q3 + 1.5 * iqr))
    if outliers.sum() > 0:
        return True
    return False

In [35]:
for c in num_list:
    print(verify_outliers(dirty,c))

True
True
True
True
True
True
True
True


In [36]:
dirty[dirty['Pregnancies']==-2]

,Pregnancies,Glucose,Blood Pressure,Skin Thickness,Insulin,Body Mass Index,Diabetes Pedigree Function,Age,Outcome,BMI Category,Clinic Region,Care Path,Patient Segment
63,-2,141.0,58.0,34.0,128.0,25.4,0.699,24.0,0,obese,South,high-risk,adult


In [37]:
dirty['Pregnancies'] = dirty['Pregnancies'].replace(-2, 2)
dirty.iloc[63]

Pregnancies                           2
Glucose                           141.0
Blood Pressure                     58.0
Skin Thickness                     34.0
Insulin                           128.0
Body Mass Index                    25.4
Diabetes Pedigree Function        0.699
Age                                24.0
Outcome                               0
BMI Category                      obese
Clinic Region                     South
Care Path                     high-risk
Patient Segment                   adult
Name: 63, dtype: object

## 7. Preparing the dataset for Ingestion Pipelines

In [38]:
dirty.sample(5)

,Pregnancies,Glucose,Blood Pressure,Skin Thickness,Insulin,Body Mass Index,Diabetes Pedigree Function,Age,Outcome,BMI Category,Clinic Region,Care Path,Patient Segment
489,8,194.0,80.0,29.0,126.0,26.1,0.551,67.0,0,overweight,South,routine,senior
767,1,119.0,70.0,31.0,126.0,30.4,0.315,23.0,1,normal,South,routine follow-up,adult
734,2,119.0,75.0,29.0,126.0,23.3,0.560,53.0,0,normal,North,urgent monitoring,senior
581,6,109.0,60.0,27.0,126.0,32.4,0.206,27.0,0,unknown,South,high-risk,adult
480,3,158.0,70.0,30.0,328.0,35.5,0.344,35.0,1,obese,North,high-risk,adult


In [39]:
dirty.to_csv('diabetes-clean.csv', index=False)